In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [3]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

model = ChatOllama(model="llama3.2", temperature=0.3)

# create subagents

subagent_1 = create_agent(
    model=model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model=model,
    tools=[square]
)

## Calling subagents

In [4]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number. Enter a simple float as input NOTHING ELSE"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [5]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='049f2569-a9c7-47b7-8a39-21691b26b14e'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-09-14T08:49:19.614671425Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5423060783, 'load_duration': 5139191515, 'prompt_eval_count': 237, 'prompt_eval_duration': 82831467, 'eval_count': 20, 'eval_duration': 190205106, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a09f1b-2fa6-7d20-a560-815e27bb1165-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': '56922836-900d-43c5-99cc-fa1bea52d0ac', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 237, 'output_tokens': 20, 'total_tokens': 257}),
              ToolMessage(content='The square root of 456.0 is 21.354156504062622.', name='call_subagent_1', id='849b9b2a-0b71-4340-b4d